# AstroGuide — Phase 2 Demo (Google Colab)

This notebook demonstrates the two LangChain tools wired to a ReAct agent with memory,
plus Pydantic-parsed structured output.

**Prerequisites**: Add your `GOOGLE_API_KEY` to Colab Secrets (🔑 icon in the left sidebar).

In [1]:
!pip install -r requirements.txt -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
# Cell 2 — Set API key from Colab Secrets & add project to path
import os, sys
from google.colab import userdata
from IPython.display import display, Markdown

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Ensure src/ is importable
if "/content/AstroGuide" not in sys.path:
    sys.path.insert(0, "/content/AstroGuide")

from src.agent import ask, parse_chart_summary
from src.schemas import ChartSummary

# Helper: Gemini returns content as a list of blocks with extras/signature.
# This extracts just the readable text.
def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return '\n'.join(b.get('text', '') for b in content if isinstance(b, dict))
    return str(content)

print("Imports OK ✅")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Cell 3 — Demo question (triggers BOTH tools: birth chart + numerology)
result = ask(
    "My name is Aditya, born 1998-05-14 at 09:30 in Mumbai, India. "
    "What's my Moon sign and my life path number?"
)

# Render the final AI response as formatted markdown
display(Markdown(extract_text(result["messages"][-1].content)))

In [ ]:
# Cell 4 — Raw message trace (SCREENSHOT THIS — proof of tool_calls)
for i, msg in enumerate(result["messages"]):
    print(f"\n{'='*60}")
    print(f"Message {i}: {msg.__class__.__name__}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  tool_calls: {msg.tool_calls}")
    if hasattr(msg, 'content'):
        text = extract_text(msg.content)
        print(f"  content: {text[:500]}")

In [ ]:
# Cell 5 — Pydantic structured output demo
from src.tools import CHART_CACHE

# Build a prompt from cached chart data
chart_data = CHART_CACHE.get("Aditya", {})
prompt = (
    f"Extract a ChartSummary from this Vedic birth chart data.\n"
    f"Name: Aditya\n"
    f"Chart positions: {chart_data}\n"
    f"Return the name, moon_sign, ascendant, and sun_sign."
)

summary: ChartSummary = parse_chart_summary(prompt)
print(f"\nPydantic-parsed output:")
print(f"  Type : {type(summary).__name__}")
print(f"  Name : {summary.name}")
print(f"  Moon : {summary.moon_sign}")
print(f"  Asc  : {summary.ascendant}")
print(f"  Sun  : {summary.sun_sign}")